In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

df = pd.read_parquet('demo_1000.parquet')
print(f"数据形状: {df.shape}")
print(f"\n缺失值:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:
# 标签分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label_type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('标签类型分布')
axes[0].set_xlabel('Label Type')
axes[0].set_ylabel('Count')

axes[1].pie(df['label_type'].value_counts(), labels=df['label_type'].value_counts().index, autopct='%1.1f%%')
axes[1].set_title('标签类型占比')

plt.tight_layout()
plt.show()

print(f"\n标签分布:\n{df['label_type'].value_counts()}")
print(f"正样本比例: {(df['label_type'] > 0).mean():.2%}")

In [ ]:
# 用户和物品统计
print(f"唯一用户数: {df['user_id'].nunique()}")
print(f"唯一物品数: {df['item_id'].nunique()}")
print(f"平均每用户交互次数: {len(df) / df['user_id'].nunique():.2f}")
print(f"平均每物品被交互次数: {len(df) / df['item_id'].nunique():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

user_counts = df['user_id'].value_counts()
axes[0].hist(user_counts, bins=30, color='coral', edgecolor='black')
axes[0].set_title('用户交互次数分布')
axes[0].set_xlabel('交互次数')
axes[0].set_ylabel('用户数')

item_counts = df['item_id'].value_counts()
axes[1].hist(item_counts, bins=30, color='lightgreen', edgecolor='black')
axes[1].set_title('物品被交互次数分布')
axes[1].set_xlabel('被交互次数')
axes[1].set_ylabel('物品数')

plt.tight_layout()
plt.show()

In [ ]:
# 时间分布
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['hour'] = df['datetime'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['hour'].value_counts().sort_index().plot(kind='line', ax=axes[0], marker='o', color='purple')
axes[0].set_title('小时分布')
axes[0].set_xlabel('小时')
axes[0].set_ylabel('交互数')
axes[0].grid(True, alpha=0.3)

hour_label = df.groupby('hour')['label_type'].mean()
hour_label.plot(kind='line', ax=axes[1], marker='s', color='orange')
axes[1].set_title('不同小时的平均标签值')
axes[1].set_xlabel('小时')
axes[1].set_ylabel('平均标签值')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 稀疏特征基数分析
user_int_cols = [col for col in df.columns if col.startswith('user_int_feats_')]
item_int_cols = [col for col in df.columns if col.startswith('item_int_feats_')]

# 过滤掉包含数组的列
user_int_cols = [col for col in user_int_cols if not isinstance(df[col].iloc[0], np.ndarray)]
item_int_cols = [col for col in item_int_cols if not isinstance(df[col].iloc[0], np.ndarray)]

user_cardinality = {col: df[col].nunique() for col in user_int_cols[:10]}
item_cardinality = {col: df[col].nunique() for col in item_int_cols}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(list(user_cardinality.keys()), list(user_cardinality.values()), color='skyblue')
axes[0].set_title('用户稀疏特征基数 (前10个)')
axes[0].set_xlabel('唯一值数量')

axes[1].barh(list(item_cardinality.keys()), list(item_cardinality.values()), color='lightcoral')
axes[1].set_title('物品稀疏特征基数')
axes[1].set_xlabel('唯一值数量')

plt.tight_layout()
plt.show()

print(f"\n用户稀疏特征总数: {len(user_int_cols)}")
print(f"物品稀疏特征总数: {len(item_int_cols)}")

In [ ]:
# 稠密特征分布
user_dense_cols = [col for col in df.columns if col.startswith('user_dense_feats_')]
user_dense_cols = [col for col in user_dense_cols if not isinstance(df[col].iloc[0], np.ndarray)]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, col in enumerate(user_dense_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='teal', alpha=0.7, edgecolor='black')
    axes[i].set_title(col.replace('user_dense_feats_', 'dense_'))
    axes[i].set_xlabel('值')
    axes[i].set_ylabel('频数')

plt.tight_layout()
plt.show()

print("\n稠密特征统计:")
print(df[user_dense_cols].describe())

In [ ]:
# 序列特征分析
domain_prefixes = ['domain_a_seq_', 'domain_b_seq_', 'domain_c_seq_', 'domain_d_seq_']
seq_lengths = {}

for prefix in domain_prefixes:
    seq_cols = [col for col in df.columns if col.startswith(prefix)]
    domain_name = prefix.replace('_seq_', '').replace('_', ' ').title()
    seq_lengths[domain_name] = len(seq_cols)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(seq_lengths.keys(), seq_lengths.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
ax.set_title('各Domain序列特征数量')
ax.set_ylabel('序列长度')
ax.set_xlabel('Domain')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n序列特征统计:")
for domain, length in seq_lengths.items():
    print(f"{domain}: {length}个特征")

In [ ]:
# 特征与标签的相关性分析（选取部分关键特征）
key_features = user_int_cols[:5] + item_int_cols[:5] + user_dense_cols[:5]
key_features = [f for f in key_features if f in df.columns]

if key_features:
    correlations = df[key_features + ['label_type']].corr()['label_type'].drop('label_type').sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['green' if x > 0 else 'red' for x in correlations.values]
    correlations.plot(kind='barh', ax=ax, color=colors, alpha=0.7)
    ax.set_title('关键特征与标签的相关性')
    ax.set_xlabel('相关系数')
    ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.show()
    
    print("\n相关性Top5:")
    print(correlations.head())
else:
    print("没有可用的标量特征进行相关性分析")